# SmolVLA Replication Analysis

Reads:
- [`results_smolvla/rollouts_smolvla.db`](results_smolvla/rollouts_smolvla.db) from [`test_smolvla_jennifer.ipynb`](test_smolvla_jennifer.ipynb)
- [`results_v2/rollouts_v2.db`](results_v2/rollouts_v2.db) (π0.5, read-only for cross-model comparison on the **same episodes**)

**Scope (actual data):**
- **80/80** `vanilla` SmolVLA rollouts on the fixed v2 slice — reportable baseline.
- **16** `pnp_uncertainty_only` rollouts (step config `[4,5]` only; partial run).
- Do **not** compare aggregate P&P success rate to the 80-episode vanilla rate on different episode sets.

**Report focus:** (A) SmolVLA vanilla baseline vs π0.5 on the slice; (B) paired detector transfer on the 16 shared episodes; (C) failure taxonomy on that subset.

Sections: **0** coverage · **A** vanilla baseline · **B** paired outcomes · **C** detector · **D** taxonomy · **E** report tables


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sqlite3, json, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from IPython.display import display, Markdown

SHARED = '/content/drive/MyDrive/cs159-sp26'
PI05_RESULTS = f'{SHARED}/results_v2'
PI05_DB = f'{PI05_RESULTS}/rollouts_v2.db'
SMOLVLA_RESULTS = f'{SHARED}/results_smolvla'
SMOLVLA_DB = f'{SMOLVLA_RESULTS}/rollouts_smolvla.db'
FIGURES_DIR = f'{SMOLVLA_RESULTS}/figures_smolvla'
FINAL_FIGURES_DIR = f'{FIGURES_DIR}/final'

for path, fallback in [(SMOLVLA_DB, 'results_smolvla/rollouts_smolvla.db'),
                       (PI05_DB, 'results_v2/rollouts_v2.db')]:
    if not os.path.isfile(path):
        local = Path(fallback)
        if local.is_file():
            if 'smolvla' in fallback:
                SMOLVLA_RESULTS = str(local.parent)
                SMOLVLA_DB = str(local)
            else:
                PI05_RESULTS = str(local.parent)
                PI05_DB = str(local)

os.makedirs(FINAL_FIGURES_DIR, exist_ok=True)

def save_fig(name, final=True):
    d = FINAL_FIGURES_DIR if final else FIGURES_DIR
    os.makedirs(d, exist_ok=True)
    path = os.path.join(d, f'final_smolvla_{name}.png' if final else f'smolvla_{name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Wrote {path}')

FINAL_STEP_CONFIGS = [(2, 3), (3, 4), (4, 5)]
PRIMARY_PNP_CONFIG = '[4, 5]'  # only step config with n=16 in partial run
EPISODE_KEYS = ['suite', 'task_idx', 'episode_idx', 'init_state_hash']
INSTABILITY_COLS = ['action_delta_l2_mean', 'action_delta_l2_max', 'action_var_mean',
                    'gripper_flip_count', 'gripper_flip_rate', 'chunk_disagreement_mean']

con_smol = sqlite3.connect(SMOLVLA_DB)
smol_rollouts = pd.read_sql('SELECT * FROM rollouts', con_smol)
con_pi = sqlite3.connect(PI05_DB)
pi05_rollouts = pd.read_sql('SELECT * FROM rollouts', con_pi)

smol_df = smol_rollouts[smol_rollouts['final_eval_slice'] == 1].copy()
pi05_df = pi05_rollouts[pi05_rollouts['final_eval_slice'] == 1].copy()
print(f'SmolVLA slice rows: {len(smol_df)}')
print(f'π0.5 slice rows:    {len(pi05_df)}')


---
## Section 0: Coverage and paired subset

Same **80 episodes** as π0.5 v2 (`libero_goal` + `libero_spatial`, top-8 failure-prone tasks).

| Run | Status | Use in report |
|-----|--------|---------------|
| SmolVLA `vanilla` | 80/80 episodes | **Primary** baseline SR + instability |
| SmolVLA `pnp_uncertainty_only` | 16 episodes, config `[4,5]` only | **Paired** detector / taxonomy only |
| π0.5 (read-only) | full v2 matrix on all 80 | Cross-model detector on **same 16** episodes |

**Hypothesis (narrow):** On episodes where we measured P&P uncertainty for both models, does SmolVLA show the same *direction* of U↔failure association as π0.5? With $n{=}16$, treat detector AUC as exploratory, not headline.


In [ ]:
def collapse_step_configs(df, success_agg='max'):
    if df.empty:
        return df
    agg = {'success': success_agg}
    for col in ['u_mean_episode', 'n_steps'] + INSTABILITY_COLS:
        if col in df.columns:
            agg[col] = 'mean'
    return df.groupby(['method'] + EPISODE_KEYS, as_index=False).agg(agg)


def describe_coverage(df, label):
    methods = df['method'].value_counts().to_dict()
    print(f'=== {label} ===')
    print(f'  Unique episodes: {df[EPISODE_KEYS].drop_duplicates().shape[0]}')
    print(f'  Methods: {methods}')
    pnp = df[df['method'] == 'pnp_uncertainty_only']
    if len(pnp):
        print('  P&P step configs:', pnp['pnp_step_indices'].value_counts().to_dict())
    if 'vanilla' in methods:
        v = df[df['method'] == 'vanilla']
        print(f'  vanilla SR ({len(v)} eps): {v["success"].mean():.1%}')

describe_coverage(smol_df, 'SmolVLA')
describe_coverage(pi05_df, 'π0.5')


def build_paired_subset(smol, pi05, pnp_step_config=PRIMARY_PNP_CONFIG):
    """Episodes with SmolVLA vanilla + pnp_uncertainty_only at a fixed step config."""
    van = smol[smol['method'] == 'vanilla'][EPISODE_KEYS + ['success'] + INSTABILITY_COLS]
    van = van.rename(columns={'success': 'vanilla_success',
                              **{c: f'vanilla_{c}' for c in INSTABILITY_COLS}})

    pnp = smol[(smol['method'] == 'pnp_uncertainty_only') & (smol['pnp_step_indices'] == pnp_step_config)].copy()
    pnp = pnp[EPISODE_KEYS + ['success', 'u_mean_episode'] + INSTABILITY_COLS]
    pnp = pnp.rename(columns={
        'success': 'smol_pnp_success',
        'u_mean_episode': 'smol_u',
        **{c: f'smol_pnp_{c}' for c in INSTABILITY_COLS},
    })

    paired = van.merge(pnp, on=EPISODE_KEYS, how='inner')
    if paired.empty:
        return paired

    pi_van = pi05[pi05['method'] == 'vanilla'][EPISODE_KEYS + ['success']]
    pi_van = pi_van.rename(columns={'success': 'pi05_vanilla_success'})
    pi_pnp = pi05[(pi05['method'] == 'pnp_uncertainty_only') & (pi05['pnp_step_indices'] == pnp_step_config)]
    pi_pnp = pi_pnp[EPISODE_KEYS + ['success', 'u_mean_episode', 'action_delta_l2_mean']]
    pi_pnp = pi_pnp.rename(columns={'success': 'pi05_pnp_success', 'u_mean_episode': 'pi05_u',
                                    'action_delta_l2_mean': 'pi05_pnp_jerk'})

    paired = paired.merge(pi_van, on=EPISODE_KEYS, how='left')
    paired = paired.merge(pi_pnp, on=EPISODE_KEYS, how='left')
    return paired

paired_df = build_paired_subset(smol_df, pi05_df)
print(f'\nPaired subset (SmolVLA vanilla + pnp [{PRIMARY_PNP_CONFIG}]): n={len(paired_df)}')
if not paired_df.empty:
    display(paired_df[EPISODE_KEYS + ['vanilla_success', 'smol_pnp_success', 'smol_u',
                                      'pi05_vanilla_success', 'pi05_pnp_success', 'pi05_u']].round(4))


---
## Section A: SmolVLA vanilla baseline (80 episodes)

Headline numbers for the report: success rate and action instability on the **full** fixed slice. Compare π0.5 `vanilla` on the same 80 episodes (not P&P methods).


In [ ]:
vanilla_smol = smol_df[smol_df['method'] == 'vanilla'].copy()
vanilla_pi = pi05_df[pi05_df['method'] == 'vanilla'].copy()

baseline_rows = []
for label, sub in [('SmolVLA', vanilla_smol), ('π0.5', vanilla_pi)]:
    row = dict(model=label, n_episodes=len(sub), success_rate=sub['success'].mean())
    for col in INSTABILITY_COLS:
        if col in sub.columns and sub[col].notna().any():
            row[col] = sub[col].mean()
    baseline_rows.append(row)

baseline_df = pd.DataFrame(baseline_rows)
print('=== Vanilla baseline on full v2 slice (80 episodes) ===')
display(baseline_df.round(4))
baseline_df.to_csv(os.path.join(SMOLVLA_RESULTS, 'final_smolvla_vanilla_baseline.csv'), index=False)

fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(baseline_df['model'], baseline_df['success_rate'], color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Success rate')
ax.set_ylim(0, 1)
ax.set_title('Vanilla success on v2 slice (n=80 each)')
for i, sr in enumerate(baseline_df['success_rate']):
    ax.text(i, sr + 0.02, f'{sr:.1%}', ha='center', fontsize=10)
save_fig('vanilla_baseline_sr')
plt.show()

if vanilla_smol['action_delta_l2_mean'].notna().any():
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.hist(vanilla_smol['action_delta_l2_mean'], bins=15, alpha=0.7, label='SmolVLA', color='#4C72B0')
    ax.hist(vanilla_pi['action_delta_l2_mean'], bins=15, alpha=0.5, label='π0.5', color='#DD8452')
    ax.set_xlabel('action_delta_l2_mean')
    ax.set_ylabel('Episode count')
    ax.set_title('Action instability (vanilla, n=80)')
    ax.legend()
    save_fig('vanilla_instability_hist')
    plt.show()


---
## Section B: Paired outcomes on P&P subset ($n{=}16$)

On the 16 episodes with both SmolVLA `vanilla` and `pnp_uncertainty_only` at `[4,5]`, tabulate outcome transitions. This is **not** a matched-method success-rate claim — P&P measurement perturbs the rollout.


In [ ]:
if paired_df.empty:
    print('No paired episodes — run more pnp_uncertainty_only rollouts in test_smolvla Section 6c.')
else:
    n = len(paired_df)
    transition = paired_df.groupby(['vanilla_success', 'smol_pnp_success']).size().reset_index(name='count')
    print(f'=== Outcome transitions (n={n}, SmolVLA vanilla → pnp measure) ===')
    display(transition)

    paired_df['outcome'] = np.where(
        paired_df['vanilla_success'] & paired_df['smol_pnp_success'], 'both_success',
        np.where(~paired_df['vanilla_success'] & paired_df['smol_pnp_success'], 'fail_to_success',
        np.where(paired_df['vanilla_success'] & ~paired_df['smol_pnp_success'], 'success_to_fail', 'both_fail')))

    trans_summary = paired_df['outcome'].value_counts().rename_axis('outcome').reset_index(name='count')
    trans_summary['fraction'] = trans_summary['count'] / n
    print('\nTransition summary:')
    display(trans_summary.round(3))

    print(f'Vanilla SR on paired subset: {paired_df["vanilla_success"].mean():.1%}')
    print(f'P&P-measure SR on paired subset: {paired_df["smol_pnp_success"].mean():.1%}')
    print('(Different methods on same inits — SR gap is not a corrector claim.)')

    trans_summary.to_csv(os.path.join(SMOLVLA_RESULTS, 'final_smolvla_paired_transitions.csv'), index=False)


---
## Section C: Detector analysis on paired subset

Compute detector metrics on the **same 16 episodes** for SmolVLA and π0.5 (`pnp_uncertainty_only`, config `[4,5]`). Report Spearman($U$, success), ROC-AUC, and U vs. jerk correlation. With $n{=}16$ and few failures, treat AUC/F1 as noisy.


In [ ]:
try:
    from sklearn.metrics import roc_auc_score, average_precision_score
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])
    from sklearn.metrics import roc_auc_score, average_precision_score


def threshold_curve(y_fail, scores):
    thresholds = np.linspace(scores.min(), scores.max(), 100)
    best = {'f1': -1}
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (y_fail == 1)).sum()
        fp = ((pred == 1) & (y_fail == 0)).sum()
        fn = ((pred == 0) & (y_fail == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        if f1 > best.get('f1', -1):
            best = {'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1}
    return best


def detector_row(model, u, success, jerk=None):
    u = np.asarray(u, dtype=float)
    success = np.asarray(success, dtype=int)
    y_fail = 1 - success
    row = {
        'model': model,
        'n': len(u),
        'n_failures': int(y_fail.sum()),
        'success_rate': success.mean(),
        'spearman_u_vs_success': stats.spearmanr(u, success)[0] if len(np.unique(success)) > 1 else np.nan,
    }
    if jerk is not None and np.asarray(jerk).notna().all():
        row['spearman_u_vs_jerk'] = stats.spearmanr(u, jerk)[0]
    if y_fail.sum() > 0 and y_fail.sum() < len(y_fail):
        row['roc_auc'] = roc_auc_score(y_fail, u)
        row['pr_auc'] = average_precision_score(y_fail, u)
        row.update({f'best_{k}': v for k, v in threshold_curve(y_fail, u).items()})
    else:
        row['roc_auc'] = row['pr_auc'] = np.nan
    return row


detector_rows = []
if not paired_df.empty:
    detector_rows.append(detector_row(
        'SmolVLA', paired_df['smol_u'], paired_df['smol_pnp_success'],
        jerk=paired_df.get('smol_pnp_action_delta_l2_mean')))
    if paired_df['pi05_u'].notna().all():
        detector_rows.append(detector_row(
            'π0.5', paired_df['pi05_u'], paired_df['pi05_pnp_success']))

    pi_pnp_full = pi05_df[(pi05_df['method'] == 'pnp_uncertainty_only') &
                          (pi05_df['pnp_step_indices'] == PRIMARY_PNP_CONFIG)]
    if len(pi_pnp_full) and pi_pnp_full['success'].nunique() > 1:
        ref = detector_row('π0.5 (full slice, n=80)', pi_pnp_full['u_mean_episode'], pi_pnp_full['success'],
                           jerk=pi_pnp_full['action_delta_l2_mean'])
        detector_rows.append(ref)

detector_df = pd.DataFrame(detector_rows)
print('=== Detector metrics ===')
display(detector_df.round(4))
detector_df.to_csv(os.path.join(SMOLVLA_RESULTS, 'final_smolvla_detector_metrics.csv'), index=False)

if not paired_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, u_col, suc_col, label in [
        (axes[0], 'smol_u', 'smol_pnp_success', 'SmolVLA'),
        (axes[1], 'pi05_u', 'pi05_pnp_success', 'π0.5'),
    ]:
        m = paired_df[suc_col] == 1
        ax.scatter(paired_df.loc[m, u_col], paired_df.loc[m, suc_col], alpha=0.7, label='success')
        ax.scatter(paired_df.loc[~m, u_col], paired_df.loc[~m, suc_col], alpha=0.7, label='fail')
        ax.set_xlabel('u_mean_episode')
        ax.set_title(f'{label} (n={len(paired_df)})')
        ax.legend()
    fig.suptitle('P&P uncertainty vs outcome on paired subset', fontsize=11)
    plt.tight_layout()
    save_fig('uncertainty_success_failure')
    plt.show()

    fig, ax = plt.subplots(figsize=(5, 4))
    sc = ax.scatter(paired_df['smol_u'], paired_df['smol_pnp_action_delta_l2_mean'],
                    c=paired_df['smol_pnp_success'], cmap='coolwarm', alpha=0.8, edgecolors='k', linewidths=0.5)
    ax.set_xlabel('u_mean_episode')
    ax.set_ylabel('action_delta_l2_mean')
    ax.set_title('SmolVLA: uncertainty vs jerk (paired n=16)')
    plt.colorbar(sc, ax=ax, label='success')
    save_fig('uncertainty_vs_instability')
    plt.show()

    sub = detector_df[detector_df['model'].isin(['SmolVLA', 'π0.5'])]
    if len(sub) >= 2:
        fig, ax = plt.subplots(figsize=(4, 4))
        x = np.arange(len(sub))
        ax.bar(x, sub['roc_auc'], tick_label=sub['model'])
        ax.set_ylabel('ROC-AUC (predict failure)')
        ax.set_ylim(0, 1)
        ax.set_title(f'Detector AUC on same {len(paired_df)} episodes')
        save_fig('cross_model_detector_auc')
        plt.show()


---
## Section D: Failure taxonomy (paired subset)

Median-split on $\bar{U}$ and `action_delta_l2_mean` within the 16 P&P-measured SmolVLA episodes. Compare bucket counts to π0.5 on the same episodes.


In [ ]:
def classify_taxonomy(df, u_col, suc_col, inst_col, prefix=''):
    if df.empty:
        return pd.DataFrame()
    u_med = df[u_col].median()
    inst_med = df[inst_col].median() if inst_col in df and df[inst_col].notna().any() else 0
    rows = []
    for _, r in df.iterrows():
        hi_u = r[u_col] >= u_med
        hi_i = r.get(inst_col, 0) >= inst_med if pd.notna(r.get(inst_col)) else False
        if not r[suc_col]:
            if hi_u and hi_i:
                tax = 'high_U_high_instability_failure'
            elif not hi_u and not hi_i:
                tax = 'low_U_low_instability_failure'
            elif hi_u:
                tax = 'high_U_low_instability_failure'
            else:
                tax = 'low_U_high_instability_failure'
        else:
            tax = 'high_U_success' if hi_u else 'low_U_success'
        rows.append({**{k: r[k] for k in EPISODE_KEYS}, 'model': prefix,
                     'taxonomy': tax, 'success': r[suc_col], 'u': r[u_col]})
    return pd.DataFrame(rows)

taxonomy_parts = []
if not paired_df.empty:
    taxonomy_parts.append(classify_taxonomy(
        paired_df, 'smol_u', 'smol_pnp_success', 'smol_pnp_action_delta_l2_mean', 'SmolVLA'))
    if paired_df['pi05_u'].notna().all():
        taxonomy_parts.append(classify_taxonomy(
            paired_df, 'pi05_u', 'pi05_pnp_success', 'pi05_pnp_jerk', 'π0.5'))

taxonomy_df = pd.concat(taxonomy_parts, ignore_index=True) if taxonomy_parts else pd.DataFrame()
if not taxonomy_df.empty:
    print('=== Failure taxonomy (paired subset) ===')
    display(taxonomy_df.groupby(['model', 'taxonomy']).size().unstack(fill_value=0))
    taxonomy_df.to_csv(os.path.join(SMOLVLA_RESULTS, 'final_smolvla_failure_taxonomy.csv'), index=False)


---
## Section E: Report-ready summary

Copy numbers below into `neurips_2026.tex` Section 5.4 (SmolVLA replication placeholder).


In [ ]:
def report_summary(baseline_df, paired_df, detector_df):
    lines = ['## SmolVLA replication (for report)\n']
    if not baseline_df.empty:
        sm = baseline_df[baseline_df['model'] == 'SmolVLA'].iloc[0]
        pi = baseline_df[baseline_df['model'] == 'π0.5'].iloc[0]
        lines.append(
            f"- **Vanilla baseline (n=80):** SmolVLA {sm['success_rate']:.1%} vs "
            f"π0.5 {pi['success_rate']:.1%} on the same fixed slice."
        )
        if 'action_delta_l2_mean' in sm:
            lines.append(
                f"- **Instability:** SmolVLA mean jerk {sm['action_delta_l2_mean']:.3f} vs "
                f"π0.5 {pi['action_delta_l2_mean']:.3f}."
            )
    n_paired = len(paired_df)
    lines.append(f"- **P&P coverage:** only {n_paired}/80 episodes with `pnp_uncertainty_only` (config `[4,5]`).")
    if n_paired:
        full_sr = baseline_df.loc[baseline_df['model'] == 'SmolVLA', 'success_rate'].iloc[0]
        lines.append(
            f"- **Paired subset:** vanilla SR {paired_df['vanilla_success'].mean():.1%} "
            f"on these {n_paired} episodes (vs {full_sr:.1%} full slice)."
        )
    sub = detector_df[detector_df['model'].isin(['SmolVLA', 'π0.5'])] if not detector_df.empty else pd.DataFrame()
    if len(sub) == 2:
        sm_d = sub[sub['model'] == 'SmolVLA'].iloc[0]
        pi_d = sub[sub['model'] == 'π0.5'].iloc[0]
        lines.append(
            f"- **Detector (same {n_paired} eps, config [4,5]):** "
            f"SmolVLA ROC-AUC={sm_d.get('roc_auc', float('nan')):.2f}, "
            f"Spearman(U,success)={sm_d.get('spearman_u_vs_success', float('nan')):.2f}; "
            f"π0.5 ROC-AUC={pi_d.get('roc_auc', float('nan')):.2f}, "
            f"Spearman={pi_d.get('spearman_u_vs_success', float('nan')):.2f}."
        )
        lines.append(
            "  Interpretation: with n=16, SmolVLA does **not** replicate π0.5's strong detector (AUC≈0.89 on n=80). "
            "Report as preliminary / negative transfer, not as proof the signal is architecture-agnostic."
        )
    lines.append(
        "- **Caveat:** P&P uncertainty measures local instability under perturbation, not task correctness. "
        "Do not claim P&P improves SmolVLA success — refinement was not run."
    )
    text = '\n'.join(lines)
    display(Markdown(text))
    out = os.path.join(SMOLVLA_RESULTS, 'final_smolvla_report_summary.md')
    with open(out, 'w') as f:
        f.write(text)
    print(f'Wrote {out}')

    if not baseline_df.empty and not detector_df.empty:
        sm = baseline_df[baseline_df['model'] == 'SmolVLA'].iloc[0]
        pi = baseline_df[baseline_df['model'] == 'π0.5'].iloc[0]
        sm_d = detector_df[detector_df['model'] == 'SmolVLA']
        pi_d_sub = detector_df[detector_df['model'] == 'π0.5']
        pi_d_full = detector_df[detector_df['model'] == 'π0.5 (full slice, n=80)']
        tex_lines = [
            '% SmolVLA replication (partial P&P run)',
            r'\begin{table}[t]',
            r'  \caption{SmolVLA on the v2 slice. Detector rows use step config $[4,5]$ only.}',
            r'  \label{tab:smolvla}',
            r'  \centering',
            r'  \small',
            r'  \begin{tabular}{lrrr}',
            r'    \toprule',
            r'    Model & Episodes & Vanilla SR & ROC-AUC ($\bar{U}$ predicts fail) \\',
            r'    \midrule',
            f"    SmolVLA & 80 & {sm['success_rate']:.1%} & --- \\\\",
            f"    $\\pi_{{0.5}}$ & 80 & {pi['success_rate']:.1%} & --- \\\\",
        ]
        if len(sm_d):
            tex_lines.append(
                f"    SmolVLA (P\\&P meas.) & {int(sm_d.iloc[0]['n'])} & --- & {sm_d.iloc[0].get('roc_auc', float('nan')):.2f} \\\\"
            )
        if len(pi_d_sub):
            tex_lines.append(
                f"    $\\pi_{{0.5}}$ (same episodes) & {int(pi_d_sub.iloc[0]['n'])} & --- & {pi_d_sub.iloc[0].get('roc_auc', float('nan')):.2f} \\\\"
            )
        if len(pi_d_full):
            tex_lines.append(
                f"    $\\pi_{{0.5}}$ (full slice) & 80 & --- & {pi_d_full.iloc[0].get('roc_auc', float('nan')):.2f} \\\\"
            )
        tex_lines += [
            r'    \bottomrule',
            r'  \end{tabular}',
            r'\end{table}',
        ]
        tex = '\n'.join(tex_lines) + '\n'
        tex_path = os.path.join(SMOLVLA_RESULTS, 'final_smolvla_table.tex')
        with open(tex_path, 'w') as f:
            f.write(tex)
        print(f'Wrote {tex_path}')
        print(tex)

report_summary(baseline_df, paired_df, detector_df)
